In [1]:
#The purpose of this script is to:
#take league-specific information on player ownership and free agents from my Yahoo fantasy baseball league
#pair this with some advanced metrics from the pybaseball python package
#output a dataset with this information to help identify where opportunities may exist to pick up players where underlying numbers
#are performing well and they are available via free agency or trade with other fantasy teams
#yahoo fantasy API documentation: https://developer.yahoo.com/fantasysports/guide/

#Next - clean up manual mapping by mapping to Yahoo ID

import http.client
import pandas as pd
import numpy as np
import requests
import json
import requests
import time
from datetime import datetime
import math
import pybaseball as pb
import re
import os

yr = 2026
output_folder = os.environ['output_path'] 

#set the stats API URL
mlb_url = "https://statsapi.mlb.com/api/v1/stats"

#set some assumption variables
qual_p = .5
START_DATE = "2026-03-25"
END_DATE = "2026-10-01"

#this is for manual mapping purposes. For some reason some players dont have a fangraphs ID in the playerid_lookup function result which
#breaks my joins. There were 88 players in this situation in April 2025. 
#Hopefully the playerid_lookup function gets updated to close this gap
#but rather than continue to miss out on critical stats from certain players I created a manual mapping table that I can use
#when the fangraphs ID from the playerid_lookup function does not return a valid ID to get an ID to join with and
#reduce the players in the final data set that I don't get data for
df_fg_map = pd.read_csv(os.getcwd() + "\\FangraphID_Map.csv")


In [2]:
#set some definitions that will be used in both the pitching and hitting statcast functions 

# -------------------------
# Batted ball events only
# -------------------------

#define which descriptions result in an event that we don't want to count as a batted ball
NON_BBE_DESCRIPTIONS = {
    "foul",
    "foul_tip",
    "foul_bunt",
    "missed_bunt",
}

# A "miss" is a swing with NO contact — used to derive Contact%.
# Assumption: foul_bunt and missed_bunt are treated as misses because the
# batter failed to put the ball in play intentionally; this matches how
# Baseball Savant computes Contact%.
MISS_EVENTS = {
    "swinging_strike",
    "swinging_strike_blocked",
    "missed_bunt",
}

SWING_EVENTS = {
    "swinging_strike",
    "swinging_strike_blocked",
    "foul",
    "foul_tip",
    "foul_bunt",
    "missed_bunt",
    "hit_into_play",
    "hit_into_play_score",    # older Statcast label
}

# Outs-per-event lookup — used to derive innings pitched for xFIP
# ---------------------------------------------------------------------------
# Assumption: we count outs from the "events" column, which records the
# plate appearance outcome on the final pitch of each PA. Each event below
# contributes a known number of outs. We intentionally exclude events that
# do NOT result in outs (hits, walks, HBP, errors, etc.) — those contribute 0.
#
# Double plays (GIDP) contribute 2 outs; triple plays are extremely rare
# and contribute 3 — included for completeness.
OUTS_PER_EVENT: dict[str, int] = {
    # Single-out events
    "strikeout":              1,
    "field_out":              1,
    "force_out":              1,
    "sac_fly":                1,
    "sac_bunt":               1,
    "fielders_choice":        1,
    "caught_stealing_2b":     1,
    "caught_stealing_3b":     1,
    "caught_stealing_home":   1,
    "pickoff_caught_stealing_2b":  1,
    "pickoff_caught_stealing_3b":  1,
    "pickoff_caught_stealing_home": 1,
    "other_out":              1,
    # Two-out events
    "grounded_into_double_play": 2,
    "strikeout_double_play":     2,
    "double_play":               2,
    "sac_fly_double_play":       2,
    # Three-out events (rare)
    "triple_play":            3,
}

# Fly ball events — used to estimate expected HR for xFIP
# ---------------------------------------------------------------------------
# Assumption: Statcast "bb_type" (batted ball type) labels fly balls as
# "fly_ball". We count these as the FB denominator for xFIP's HR/FB step.
# "popup" is NOT counted as a fly ball for xFIP purposes — popups almost
# never leave the park and are typically excluded from HR/FB rate by convention.
FLY_BALL_TYPE = "fly_ball"

In [3]:
def _is_barrel(ev: pd.Series, la: pd.Series) -> pd.Series:
    """
    Return boolean Series — True where the batted ball meets barrel criteria.
 
    Parameters
    ----------
    ev : launch_speed column (mph)
    la : launch_angle column (degrees)

    The logic, per the official MLB definition:

    A minimum exit velocity of 98 mph is required. At that speed, a launch angle of 26–30° qualifies. For every mph over 98, the range expands.
    At 99 mph the window becomes 25–31° — so the model expands the lower bound by 1° and the upper bound by 1° for each additional mph. Medium
    At 116 mph the window caps at 8–50° — EV is clamped there so the window doesn't expand further. FantraxHQ

    """
    MAX_EV_CAP    = 116.0   # above this EV, angle range is fixed at 8–50°
    BASE_EV       = 98.0    # minimum EV to be a barrel
    BASE_LA_LOW   = 26.0    # lower launch angle at base EV
    BASE_LA_HIGH  = 30.0    # upper launch angle at base EV
    CAP_LA_LOW    =  8.0    # lower launch angle at cap EV
    CAP_LA_HIGH   = 50.0    # upper launch angle at cap EV

    # Clamp EV at the cap so the window doesn't expand beyond 8–50°
    ev_clamped = ev.clip(upper=MAX_EV_CAP)
 
    # How many mph above the 98 mph floor?
    mph_over_base = (ev_clamped - BASE_EV).clip(lower=0)
 
    # Expand the angle window by 1° per mph on each side
    la_low  = BASE_LA_LOW  - mph_over_base   # floor drops
    la_high = BASE_LA_HIGH + mph_over_base   # ceiling rises
 
    #return the evaluation of whether the exit velocity is over the barrel minimum and the launch angle is within the defined range
    return (ev >= BASE_EV) & (la >= la_low) & (la <= la_high)

In [4]:
#the purpose of this function is to create statcast metrics off a given dataframe of statcast data
def calculate_statcast_hit_metrics(df, SWING_EVENTS, NON_BBE_DESCRIPTIONS):

    #create the mask to filter out non-bbe events
    bbe_mask = (
        df["launch_speed"].notna() &  #must have a launch_speed populated
        ~df["description"].str.lower().isin(NON_BBE_DESCRIPTIONS) #not in one of the defined non-bbe events
    )

    #apply the mask and sum the events
    bbe = df[bbe_mask]
    total_bbe = bbe_mask.sum()

    # -------------------------
    # Barrel %
    # -------------------------

    #this definition of barrel was derived from claude code

    barrel_flags = _is_barrel(bbe["launch_speed"], bbe["launch_angle"].fillna(0)) #run the data through the function that defines a barrel
    barrel_count = barrel_flags.sum()  #get the total count
    barrel_pct   = (barrel_count / total_bbe) if total_bbe > 0 else float("nan")  #get the percent

    # -------------------------
    # Hard Hit %
    # -------------------------

    #hard hit defined as launch speed at least 95
    hard_hit_pct = (
        (bbe["launch_speed"] >= 95).mean()
        if total_bbe > 0 else float("nan")
    )

    # -------------------------
    # maxEV
    # -------------------------

    #get the max ev for this player
    max_ev = (
        bbe["launch_speed"].max()
        if total_bbe > 0 else float("nan")
    )

    # -------------------------
    # Swing decisions
    # -------------------------

    #get events that are defined as a swing
    swings = df["description"].isin(SWING_EVENTS)

    # zones 1-9 = strike zone
    in_zone = df["zone"].between(1, 9)

    #out of zone means it's not in the zone defined above
    out_zone = ~in_zone

    # O-Swing%
    o_swing_pct = (
        swings[out_zone].mean()
        if out_zone.sum() > 0 else float("nan")
    )
 
    # Z-Swing%
    z_swing_pct = (
        swings[in_zone].mean()
        if in_zone.sum() > 0 else float("nan")
    )

    #return the values we need, multiply percentages by 100 and round
    return {
        "Barrel Count": barrel_count,
        "Barrel%": round(barrel_pct * 100, 1),
        "HardHit%": round(hard_hit_pct * 100, 1),
        "maxEV": round(max_ev, 1),
        "O-Swing%": round(o_swing_pct * 100, 1),
        "Z-Swing%": round(z_swing_pct * 100, 1),
        "BBE": total_bbe,
    }

In [5]:
#This code gets the advanced statcast metrics for pitchers

def calculate_statcast_pitch_metrics(df, SWING_EVENTS, NON_BBE_DESCRIPTIONS, OUTS_PER_EVENT, FLY_BALL_TYPE, MISS_EVENTS, 
                                     fipc: float = 3.10, lg_hr_fb_rate: float = 0.105,):
    
    if len(df) == 0:
        return None

    # -----------------------------------------------------
    # Batted ball events
    # BBE = pitch with a recorded launch_speed that is NOT a foul/missed bunt.
    # See batter_metrics.py for full rationale.
    # -----------------------------------------------------

    # --- 2. Identify batted ball events (BBE) ---------------------------------
    
    bbe_mask = (
        df["launch_speed"].notna() &
        ~df["description"].str.lower().isin(NON_BBE_DESCRIPTIONS)
    )
    bbe       = df[bbe_mask]
    total_bbe = int(bbe_mask.sum())

    # -----------------------------------------------------
    # Barrel%
    # Barrels allowed / BBE. Uses same derived barrel formula as batter script.
    # launch_angle NaN on a BBE row is treated as 0° (non-barrel) to keep the
    # denominator consistent.
    # -----------------------------------------------------

    barrel_flags = _is_barrel(bbe["launch_speed"], bbe["launch_angle"].fillna(0))
    barrel_count = barrel_flags.sum()
    barrel_pct   = (barrel_count / total_bbe) if total_bbe > 0 else float("nan")

    # -----------------------------------------------------
    # HardHit%
    # -----------------------------------------------------

    hard_hit_count = (bbe["launch_speed"] >= 95).sum()
    hard_hit_pct   = (
        hard_hit_count / total_bbe if total_bbe > 0 else float("nan")
    )

    # -----------------------------------------------------
    # Contact%
    # (Swings - Misses) / Swings  =  contact swings / total swings.
    # Equivalently: 1 - SwStr%.
    #
    # Assumption: we calculate from all pitches (not just tracked-zone pitches)
    # because Contact% is a swing-outcome metric, not a zone-discipline metric.
    # -----------------------------------------------------

    desc = df["description"].str.lower()
    total_swings = desc.isin(SWING_EVENTS).sum()
    total_misses = desc.isin(MISS_EVENTS).sum()
    contact_swings = total_swings - total_misses
    contact_pct    = (
        contact_swings / total_swings if total_swings > 0 else float("nan")
    )

    # -----------------------------------------------------
    # xFIP
    # xFIP = (13 * xHR + 3 * (BB + HBP) - 2 * K) / IP + FIPc
    #
    # where xHR = fly_balls * lg_hr_fb_rate  (expected HR, not actual HR)
    #
    # Assumption: we derive each component from plate-appearance level data.
    # Statcast records the outcome of each PA on the *last pitch* of that PA
    # in the "events" column. We de-duplicate to one row per PA first so we
    # don't double-count multi-pitch ABs.
    # -----------------------------------------------------

    # Keep only rows where "events" is populated (final pitch of each PA)
    pa_df = df[df["events"].notna()].copy()
    pa_df["events_lower"] = pa_df["events"].str.lower()
 
    # Strikeouts (K)
    # Assumption: both "strikeout" and "strikeout_double_play" count as Ks.
    k = pa_df["events_lower"].isin({"strikeout", "strikeout_double_play"}).sum()
 
    # Walks (BB) — intentional walks (intent_walk) are included per FIP convention
    bb = pa_df["events_lower"].isin({"walk", "intent_walk"}).sum()
 
    # Hit by pitch (HBP)
    hbp = (pa_df["events_lower"] == "hit_by_pitch").sum()
 
    # Fly balls — from bb_type on BBE rows (not PA rows, since bb_type lives
    # on the batted ball pitch, not the final-pitch events row for non-contact PAs)
    # Assumption: we count bb_type == "fly_ball" across all BBEs.
    fb = (df["bb_type"].str.lower() == FLY_BALL_TYPE).sum()
 
    # Expected HR = fly balls * league-average HR/FB rate
    x_hr = fb * lg_hr_fb_rate
 
    # Innings pitched — derived from outs recorded
    # Assumption: we sum outs from the OUTS_PER_EVENT lookup on PA-level rows.
    # This will slightly undercount if the pitcher left mid-inning and inherited
    # runners scored, but is accurate for outs attributed to this pitcher.
    total_outs = pa_df["events_lower"].map(OUTS_PER_EVENT).fillna(0).sum()
    ip         = total_outs / 3  # convert outs to innings
 
    # xFIP calculation
    # Guard against 0 IP to avoid division by zero
    if ip > 0:
        xfip = ((13 * x_hr) + (3 * (bb + hbp)) - (2 * k)) / ip + fipc
    else:
        xfip = float("nan")

    return {
        "Barrel Count": barrel_count,
        "Barrel%": round(barrel_pct * 100, 2),
        "HardHit%": round(hard_hit_pct*100, 2),
        "Contact%": round(contact_pct * 100, 2),
        "xFIP": round(xfip, 2),
        "BBE": total_bbe
    }

In [6]:
#Step 1: Pull basic batting stats from the MLB API

#set the parameters for the connection
params = {
    "stats": "season",
    "group": "hitting",
    "season": yr,
    "playerPool": "ALL",
    "limit": 5000,
    "sportIds": 1
}

#fetch the data
r = requests.get(mlb_url, params=params)

#Convert to json
data = r.json()

#create an empty list to hold the data
rows = []

#loop through the players and pull all the individual stats we need
for player in data["stats"][0]["splits"]:

    stat = player["stat"]
    info = player["player"]
    team = player["team"]

    rows.append({
        "player_id": info["id"],
        "Name": info["fullName"],
        "Team": team["name"],

        "PA": stat.get("plateAppearances"),
        "R": stat.get("runs"),
        "HR": stat.get("homeRuns"),
        "RBI": stat.get("rbi"),
        "SO": stat.get("strikeOuts"),
        "BB": stat.get("baseOnBalls"),

        "OBP": stat.get("obp"),
        "OPS": stat.get("ops"),

        "SB": stat.get("stolenBases"),
        "CS": stat.get("caughtStealing"),

        "AB": stat.get("atBats"),
        "SF": stat.get("sacFlies"),
        "H": stat.get("hits"),
    })

#create a dataframe from the rows
df_hitter = pd.DataFrame(rows)

#convert these to float datatype
df_hitter["OBP"] = df_hitter["OBP"].astype(float)
df_hitter["OPS"] = df_hitter["OPS"].astype(float)

#Create a column that shows the percentile of plate appearances
df_hitter['PA_pct'] = df_hitter['PA'].rank(pct=True)

#Add strikeout percentage
df_hitter["K%"] = round(df_hitter["SO"] / df_hitter["PA"], 4)

#calculate BABIP
df_hitter["BABIP"] = round(
    (df_hitter["H"] - df_hitter["HR"])
    /
    (df_hitter["AB"] - df_hitter["SO"] - df_hitter["HR"] + df_hitter["SF"])
,4)


#calculate OBP+
lg_obp = df_hitter[df_hitter['PA_pct'] >= (1-qual_p)]["OBP"].astype(float).mean()

df_hitter["OBP+"] = round(
    100 * df_hitter["OBP"].astype(float) / lg_obp
,2)

#Calculate BABIP+
lg_babip = df_hitter[df_hitter['PA_pct'] >= (1-qual_p)]["BABIP"].mean()

df_hitter["BABIP+"] = round(100 * df_hitter["BABIP"] / lg_babip,2)

#calculate OPS+
lg_ops = df_hitter[df_hitter['PA_pct'] >= (1-qual_p)]["OPS"].astype(float).mean()

df_hitter["OPS+"] = round(
    100 * df_hitter["OPS"].astype(float) / lg_ops
,2)

In [7]:
#pull all the statcast data into one dataframe. Previously this was done player by player and took significantly longer
sc_all = pb.statcast(
            START_DATE,
            END_DATE
        )

This is a large query, it may take a moment to complete


100%|██████████| 191/191 [04:51<00:00,  1.52s/it]


In [8]:
#this section gets the advanced statcast metrics for hitters
#NEW METHOD - NEED TO TEST


# ---------------------------------------------------
# FINAL DATAFRAME
# ---------------------------------------------------

df_sc = (
    sc_all.groupby("batter")
    .apply(lambda g: pd.Series(calculate_statcast_hit_metrics(g, SWING_EVENTS, NON_BBE_DESCRIPTIONS)))
    .reset_index()
    .rename(columns={"batter": "player_id"})
)


df_sc.head()

,player_id,Barrel Count,Barrel%,HardHit%,maxEV,O-Swing%,Z-Swing%,BBE
0,457705,3.0,5.8,28.8,106.0,20.5,68.9,52.0
1,467793,0.0,0.0,43.8,107.2,35.7,68.0,16.0
2,500743,10.0,4.9,35.9,109.8,32.3,68.7,206.0
3,502054,1.0,6.7,53.3,108.8,24.2,60.7,15.0
4,502671,24.0,9.2,35.9,109.8,32.2,65.5,262.0


In [9]:
#Pull basic pitching stats from the MLB API

#set the parameters for the connection
params = {
    "stats": "season",
    "group": "pitching",
    "season": yr,
    "playerPool": "ALL",
    "limit": 5000,
    "sportIds": 1
}

#fetch the data
r = requests.get(mlb_url, params=params)

#Convert to json
data = r.json()

#create an empty list to hold the data
rows = []

#loop through each item 
for p in data["stats"][0]["splits"]:

    #get the stat description and player ID
    stat = p["stat"]
    player = p["player"]

    #get each 
    row = {
        "player_id": player["id"],
        "Name": player["fullName"],

        # Traditional stats
        "W": stat.get("wins"),
        "ERA": stat.get("era"),
        "WHIP": stat.get("whip"),
        "GS": stat.get("gamesStarted"),
        "SO": stat.get("strikeOuts"),
        "IP": stat.get("inningsPitched"),
        "SV": stat.get("saves"),
        "HLD": stat.get("holds"),
        "HR": stat.get("homeRuns"),
        "BB": stat.get("baseOnBalls"),

        # Needed for calculations
        "H": stat.get("hits"),
        "BF": stat.get("battersFaced"),
        "AB": stat.get("atBats"),
        "SF": stat.get("sacFlies"),
        "HBP": stat.get("hitBatsmen"),
    }

    #append the data together
    rows.append(row)

#convert the list to a dataframe
df_pitcher = pd.DataFrame(rows)

#convert these to float data type
df_pitcher['ERA'] = df_pitcher['ERA'].replace('-.--', 0).astype(float)
df_pitcher['WHIP'] = df_pitcher['WHIP'].replace('-.--', 0).astype(float)

#Calculate additional metrics

# -----------------------------------------------------
# K/9
# -----------------------------------------------------

#Create an Innings Pitched metric with accurate decimals
df_pitcher["IP_float"] = (
    df_pitcher["IP"]
    .astype(str)
    .str.replace(".1", ".333")
    .str.replace(".2", ".667")
    .astype(float)
)

#Create a column that shows the percentile of innings pitched
df_pitcher['IP_pct'] = df_pitcher['IP_float'].rank(pct=True)

#calculate K/9
df_pitcher["K/9"] = round(
    df_pitcher["SO"] * 9 / df_pitcher["IP_float"]
,2)

# -----------------------------------------------------
# K/BB
# -----------------------------------------------------

df_pitcher["K/BB"] = round(
    df_pitcher["SO"] / df_pitcher["BB"]
,2)

# -----------------------------------------------------
# BABIP
# -----------------------------------------------------

# BABIP = (H - HR) / (AB - SO - HR + SF)

df_pitcher["BABIP"] = round(
    (df_pitcher["H"] - df_pitcher["HR"])
    /
    (
        df_pitcher["AB"]
        - df_pitcher["SO"]
        - df_pitcher["HR"]
        + df_pitcher["SF"]
    )
,4)

# -----------------------------------------------------
# BABIP+
# -----------------------------------------------------

lg_babip = df_pitcher[df_pitcher['IP_pct'] >= (1-qual_p)]["BABIP"].mean()

df_pitcher["BABIP+"] = round(
    100 * df_pitcher["BABIP"] / lg_babip
,2)

# -----------------------------------------------------
# FIP
# -----------------------------------------------------

# Approximate yearly constant
FIP_CONSTANT = 3.1

df_pitcher["FIP"] = round(
    (
        13 * df_pitcher["HR"]
        + 3 * (df_pitcher["BB"] + df_pitcher["HBP"])
        - 2 * df_pitcher["SO"]
    )
    / df_pitcher["IP_float"]
,2) + FIP_CONSTANT

In [10]:
#this section gets the advanced statcast metrics for pitchers
#NEW METHOD - NEED TO TEST

# ---------------------------------------------------
# FINAL DATAFRAME
# ---------------------------------------------------

df_scp = (
    sc_all.groupby("pitcher")
    .apply(lambda g: pd.Series(calculate_statcast_pitch_metrics(g, SWING_EVENTS, NON_BBE_DESCRIPTIONS,OUTS_PER_EVENT, FLY_BALL_TYPE,MISS_EVENTS, FIP_CONSTANT)))
    .reset_index()
    .rename(columns={"pitcher": "player_id"})
)

df_scp.head()

,player_id,Barrel Count,Barrel%,HardHit%,Contact%,xFIP,BBE
0,434378,4.0,13.79,44.83,81.40,4.47,29.0
1,445276,9.0,8.57,38.10,75.42,3.68,105.0
2,453286,19.0,7.92,33.75,81.53,4.32,240.0
3,455119,6.0,11.11,44.44,86.36,3.84,54.0
4,471911,4.0,6.56,40.98,83.21,3.93,61.0


In [11]:
#merge the basic and advanced metrics for hitters, drop name from the statcast dataframe
df_batter_stat_final = df_hitter.merge(df_sc, how='left',on='player_id').reset_index()

#Drop some additional unnecessary columns
df_batter_stat_final.drop(columns=['index'], inplace=True)

#merge the basic and advanced metrics for pitchers, drop name from the statcast dataframe
df_pitcher_stat_final = df_pitcher.merge(df_scp, how='left',on='player_id').reset_index()

#Drop some additional unnecessary columns
df_pitcher_stat_final.drop(columns=['index'], inplace=True)

In [12]:
#This module is establishing a connection to the Yahoo Fantasy Sports API and setting up variables for my league and team
#https://yahoo-fantasy-api.readthedocs.io/en/latest/yahoo_fantasy_api.html - references

#Oauth instructions
#https://pypi.org/project/yahoo-oauth/
#https://developer.yahoo.com/apps/2YZSzEV6/

#Import any packages we need
from yahoo_oauth import OAuth2
import yahoo_fantasy_api as yfa

#connect to yahoo API, this environment variable points to a json file that has all the API Oauth2 authentication information
sc = OAuth2(None, None, from_file = os.environ['API_OAUTH_YFNTSY_F'])

#get the game object, we are looking at mlb information
gm = yfa.Game(sc, 'mlb')

#Set the team and league specific variables
league_id = gm.game_id() + '.l.' + str(os.environ['YFNTSY_LGID'])
teamkey = league_id + os.environ['YFNTSY_TMID'] #My team

#create a variable to get the current league, then set a variable for my team's key
lg = gm.to_league(league_id)

[2026-09-27 10:29:46,260 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-09-27 10:29:46,265 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 505.11576199531555
[2026-09-27 10:29:46,267 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


In [13]:
#Step 2a: Get the free agent data

#set some variables to start getting all of the free agents
#Update 6/16/23 - added "OF" because Seiya Suzuki was not showing when I pulled batters or "Util" players. Didn't notice other players missing for now
lsPos = ['SP','B','OF']
df_fa_all = pd.DataFrame()
lsFA = []

#Loop through the different positions
for l in lsPos:
    
    #Get the json of the free agents and put it in a dataframe
    fa = lg.free_agents(l)
    
    #append the new player info to the existing list
    lsFA = lsFA + fa

#convert the player data to a dataframe
df_fa_all = pd.DataFrame.from_records(lsFA)

#print a sample to see what we have
df_fa_all.head()

,player_id,name,editorial_team_abbr,position_type,eligible_positions,percent_owned,status
0,7590,Justin Verlander,DET,P,[SP],5,
1,8193,Max Scherzer,TOR,P,[SP],21,
2,8652,Martín Pérez,ATL,P,"[SP, IL]",30,IL15
3,9007,Nathan Eovaldi,TEX,P,[SP],70,
4,9123,Jameson Taillon,TOR,P,"[SP, IL]",4,IL15


In [14]:
#The purpose of this function is to look in a manually maintained exception table for fangraph ID's
#for some reason there are some fangraphs ID's that don't show up correctly in the pybaseball playerid_lookup function, despite these ID's existing
#I created a file that maps a different ID to the fangraphs ID for the individuals I can identify in this situation
#We will always look the ID up in the playerid_lookup function, but if we can't find a valid ID there this function will try to find it

def fangraphs_m_lkup(fg_map, mlbid, return_val, fail_val, yahoo_id):
    
    #try to look up the record on the input yahoo ID. This won't exist for most records but it will help us resolve duplicates
    #when players have the same name
    ry = fg_map[fg_map['IDYahoo'] == yahoo_id]

    #look up the record for the input mlb ID
    rm = fg_map[fg_map['IDmlb'] == mlbid]

    #if there is any record there, return the fangraph ID, otherwise return whatever value we wanted if it failed
    if ry.shape[0] > 0:
        return(ry.iloc[0][return_val])  #return value based on yahoo ID
    elif rm.shape[0] > 0:
        return(rm.iloc[0][return_val]) #return value based on mlb ID
    else:
        return(fail_val)

    

In [15]:
#Step 2b: modify the free agent data once we have defined it in the previous step

#Get rid of the duplicates
df_fa_all = df_fa_all.drop_duplicates(subset=['player_id'])

#Add some new columns that we will populate below
df_fa_all['name'] = None
df_fa_all['owner'] = 'FA'

#The index variable is jacked up because we stitched all of these datafames together. This fixes it.
df_fa_all = df_fa_all.reset_index()

#Loop through each row of the dataframe, pull each player ID and populate the addiitonal player attribute columns we created above
for index, row in df_fa_all.iterrows():
    
    #get the player detail using the player ID based on what iteration of the loop we are on
    pdtl = lg.player_details(row.loc['player_id'])[0]

    #update the player detail in the appropriate column in our orignal dataframe
    df_fa_all.at[index, 'name'] = pdtl['name']['full']

    #We need to slow this down to not make too many calls in a row but trying to do that less frequently
    if index/50 == math.floor(index/50) and index > 0:
        
        #We need to slow this down to not make too many calls in a row but trying to do that less frequently
        time.sleep(50)
        
        #let the user know what's going on
        print("Still going ... we are at loop number {} of {}".format(index,df_fa_all.shape[0])) #see how we are doing
        
#Let them know we are finally done
print("done with loop, Victory!")

#drop this index column so it aligns with the 'owned' dataframe below
df_fa_all = df_fa_all.drop('index',axis=1)

#split into pitcher and batter players
df_fa_p = df_fa_all[df_fa_all['position_type'] == 'P']
df_fa_b = df_fa_all[df_fa_all['position_type'] == 'B']

#print a sample to see what we have
df_fa_p.head()

Still going ... we are at loop number 50 of 598
Still going ... we are at loop number 100 of 598
Still going ... we are at loop number 150 of 598
Still going ... we are at loop number 200 of 598
Still going ... we are at loop number 250 of 598
Still going ... we are at loop number 300 of 598
Still going ... we are at loop number 350 of 598
Still going ... we are at loop number 400 of 598
Still going ... we are at loop number 450 of 598
Still going ... we are at loop number 500 of 598
Still going ... we are at loop number 550 of 598
done with loop, Victory!


,player_id,name,editorial_team_abbr,position_type,eligible_positions,percent_owned,status,owner
0,7590,Justin Verlander,DET,P,[SP],5,,FA
1,8193,Max Scherzer,TOR,P,[SP],21,,FA
2,8652,Martín Pérez,ATL,P,"[SP, IL]",30,IL15,FA
3,9007,Nathan Eovaldi,TEX,P,[SP],70,,FA
4,9123,Jameson Taillon,TOR,P,"[SP, IL]",4,IL15,FA


In [16]:
#This function centralizes mapping a data set from Yahoo to stats from PyBaseball for a pitcher
#I put this into it's own function for 2 reasons.
#one is because I needed to use the same mechanism in 2 independent locations, one for free agents and one for owned players
#the second is since this has to map based on name if I run into something not mapping correctly I can force the mapping in the function. 
def YahooPlayerMap(df, yr):
    
    #Transfer the data in the input dataframe to a new variable
    df_final = df.copy()
    
    #remove any middle initials with this regex pattern: [A-z]+\.
    df_final['name'] = df_final['name'].str.replace(r'\s+[A-z]+\.', '', regex=True)
    
    #create a dataframe with the split names so we can do some cleanup before it is integrated to the broader dataframe
    df_nm = df_final['name'].str.split(' ', expand = True)

    #do some cleanup
    df_nm.fillna('',inplace = True) #convert na's to '' because it messes up the concatenation
    df_nm = df_nm.replace('(Batter)','') #convert (Batter) to '' because Ohtani messes this up
    df_nm = df_nm.replace('(Pitcher)','') #convert (Pitcher) to '' because Ohtani messes this up

    #loop through the columns
    for i in df_nm.columns:

        #we want to ignore columns 0 and 1, so if it's above 1 that means we have a last name with spaces in it (De La Cruz) that was separated into multiple columns
        #So we want to concatenate it onto the last name with a space
        if i > 1:
            df_nm[1] = df_nm[1] + ' ' + df_nm[i]
    
    #strip out any blank spaces we just added by accident
    df_nm[1] = df_nm[1].str.strip()
    
    #now limit to only the first 2 columns which should be a clean first and last name
    df_nm = df_nm[[0,1]]
    
    #Split the player names by firstname and lastname because that's what we need to join data
    #df_final[['FirstName', 'LastName']] = df_final['name'].str.split(' ', expand = True)[[0,1]]  #old version - modified slightly for version 3.11, had to add the [[0,1]] because some names were split into a third column
    df_final[['FirstName', 'LastName']] = df_nm

    #Add dummy columns to hold the IDs
    df_final['IDfg'] = 0
    df_final['IDmlb'] = 0
    df_final['ID_FNM'] = 'NO DATA'
    df_final['ID_LNM'] = 'NO DATA'
    df_final['NumPlyr'] = 0

    #loop through each row and get the fangraphs ID for each player
    for index, row in df_final.iterrows():

        #get the player ID.  The fuzzy=true will allow us to still find the player if the name isn't an exact match
        #We then grab the value in the first row. This is risky but I don't know a better way to do it right now
        df_plyr = pb.playerid_lookup(row['LastName'],row['FirstName'], fuzzy=True)
        
        #filter for only player that played this year or last and reset the index so we all the index numbers make sense
        df_plyr = df_plyr[df_plyr.mlb_played_last.isin([yr,yr-1])]
        df_plyr = df_plyr.reset_index(drop=True)

        #create some exceptions where the name mapping isn't clean. 
        #Max Muncy - A's (2026):
        if df_final.loc[index,'yahoo_id'] == 12501:
        
            df_plyr = pd.DataFrame({'key_fangraphs': 29779, 'key_mlbam': 691777, 'name_first': 'Max', 'name_last': 'Muncy'}, index=[0])
        
        #Max Muncy - Dodgers (2026):
        elif df_final.loc[index,'yahoo_id'] == 9606:
            df_plyr = pd.DataFrame({'key_fangraphs': 13301, 'key_mlbam': 571970, 'name_first': 'Max', 'name_last': 'Muncy'}, index=[0])

        #Luis Castillo - Mariners (2026):
        elif df_final.loc[index,'yahoo_id'] == 10514:
            df_plyr = pd.DataFrame({'key_fangraphs': 15689, 'key_mlbam': 622491, 'name_first': 'Luis', 'name_last': 'Castillo'}, index=[0])

        #Michael King - Padres (2026):
        elif df_final.loc[index,'yahoo_id'] == 11661:
            df_plyr = pd.DataFrame({'key_fangraphs': 19853, 'key_mlbam': 650633, 'name_first': 'Michael', 'name_last': 'King'}, index=[0])
   
        #Grayson Rodriguez - Angels (2026):
        elif df_final.loc[index,'yahoo_id'] == 11277:
            df_plyr = pd.DataFrame({'key_fangraphs': 24492, 'key_mlbam': 680570, 'name_first': 'Grayson', 'name_last': 'Rodriguez'}, index=[0])

        #Gerrit Cole - Yankees (2026):
        elif df_final.loc[index,'yahoo_id'] == 9121:
            df_plyr = pd.DataFrame({'key_fangraphs': 13125, 'key_mlbam': 543037, 'name_first': 'Gerritt', 'name_last': 'Cole'}, index=[0])

        #Jared Jones - Pirates (2026):
        elif df_final.loc[index,'yahoo_id'] == 60907:
            df_plyr = pd.DataFrame({'key_fangraphs': 27863, 'key_mlbam': 543037, 'name_first': 'Jared', 'name_last': 'Jones'}, index=[0])

        #Adam Macko - Blue Jays (2026):
        elif df_final.loc[index,'yahoo_id'] == 12535:
            df_plyr = pd.DataFrame({'key_fangraphs': 25590, 'key_mlbam': 671936, 'name_first': 'Adam', 'name_last': 'Macko'}, index=[0])

        #A.J. Ewing - Mets (2026):
        elif df_final.loc[index,'yahoo_id'] == 63129:
            df_plyr = pd.DataFrame({'key_fangraphs': 33656, 'key_mlbam': 805999, 'name_first': 'A.J.', 'name_last': 'Ewing'}, index=[0])

        #Colt Emerson - Mariners (2026):
        elif df_final.loc[index,'yahoo_id'] == 62954:
            df_plyr = pd.DataFrame({'key_fangraphs': 33575, 'key_mlbam': 806068, 'name_first': 'Colt', 'name_last': 'Emerson'}, index=[0])
        
        #Henry Bolte - A's (2026):
        elif df_final.loc[index,'yahoo_id'] == 60309:
            df_plyr = pd.DataFrame({'key_fangraphs': 31448, 'key_mlbam': 703607, 'name_first': 'Henry', 'name_last': 'Bolte'}, index=[0])

        #Pedro Ramirez - Cubs (2026):
        elif df_final.loc[index,'yahoo_id'] == 64285:
            df_plyr = pd.DataFrame({'key_fangraphs': 29218, 'key_mlbam': 699393, 'name_first': 'Pedro', 'name_last': 'Ramirez'}, index=[0])

        #Joe Mack - Marlins (2026):
        elif df_final.loc[index,'yahoo_id'] == 12470:
            df_plyr = pd.DataFrame({'key_fangraphs': 29780, 'key_mlbam': 691788, 'name_first': 'Joe', 'name_last': 'Mack'}, index=[0])

        #Bryan Torres - Cardinals (2026):
        elif df_final.loc[index,'yahoo_id'] == 64242:
            df_plyr = pd.DataFrame({'key_fangraphs': 21464, 'key_mlbam': 663494, 'name_first': 'Bryan', 'name_last': 'Torres'}, index=[0])

        #Jesus Rodriguez - Giants (2026):
        elif df_final.loc[index,'yahoo_id'] == 61813:
            df_plyr = pd.DataFrame({'key_fangraphs': 0, 'key_mlbam': 0, 'name_first': 'Jesus', 'name_last': 'Rodriguez'}, index=[0])

        #Wade Meckler - Angels (2026):
        elif df_final.loc[index,'yahoo_id'] == 61844:
            df_plyr = pd.DataFrame({'key_fangraphs': 31490, 'key_mlbam': 685133, 'name_first': 'Wade', 'name_last': 'Meckler'}, index=[0])

        #Rikuu Nishida - White Sox (2026):
        elif df_final.loc[index,'yahoo_id'] == 64139:
            df_plyr = pd.DataFrame({'key_fangraphs': 33795, 'key_mlbam': 807747, 'name_first': 'Rikuu', 'name_last': 'Nishida'}, index=[0])

        #Nelson Velázquez - Cardinals (2026):
        elif df_final.loc[index,'yahoo_id'] == 11439:
            df_plyr = pd.DataFrame({'key_fangraphs': 23359, 'key_mlbam': 676369, 'name_first': 'Nelson', 'name_last': 'Velázquez'}, index=[0])

        #Alika Williams - A's (2026):
        elif df_final.loc[index,'yahoo_id'] == 12179:
            df_plyr = pd.DataFrame({'key_fangraphs': 27604, 'key_mlbam': 675961, 'name_first': 'Alika', 'name_last': 'Williams'}, index=[0])

        #We are using the player lookup for ID's. There are some gaps in pulling fangraph ID's but we aren't joining on that anymore
        #so I stripped out the manual mapping table, but if there ends up being issues that logic can be pulled from a prior version
        #of this code
        if df_plyr.shape[0] > 0:
            
            #set the ID values
            df_final.loc[index,['IDfg']] = df_plyr['key_fangraphs'][0]
            df_final.loc[index,['IDmlb']] = df_plyr['key_mlbam'][0]

            #These two fields are captured only to give a check to see which name we grabbed the ID for in case something doesn't look right
            df_final.loc[index, ['ID_FNM']] = df_plyr['name_first'][0]
            df_final.loc[index, ['ID_LNM']] = df_plyr['name_last'][0]
            
            #return how many rows we got in case we got multiple and had to make an assumption on who to pull
            df_final.loc[index, ['NumPlyr']] = df_plyr.shape[0]
        
    return df_final

In [17]:
#Step 3: build out ownership of players and pair with stats data. Used for 2 purposes:
#1) look at my team and how underlying numbers imply they are performing, find opportunities to hold or trade/cut
#2) look for trade candidates from other teams, underperforming but underlying numbers look strong

#pause to prevent timeouts
time.sleep(10)

#Get the result of the taken_players API call
p = lg.taken_players()

#create a list to hold the owner data
ls_o = []
cnt = 0

#Loop through all the owned players
for i in p:
    
    #add one to the count
    cnt += 1
    
    #Get the team ownership information
    o = lg.ownership([i['player_id']])
    
    #Append each individual row onto this list
    ls_o.append({'player_id' : i['player_id'],
                 'name' : i['name'], 
                 'position_type' : i['position_type'], 
                 'eligible_positions' : i['eligible_positions'], 
                 'percent_owned' : i['percent_owned'],
                 'status' : i['status'], 
                 'editorial_team_abbr': i['editorial_team_abbr'],
                 'owner' : o[str(i['player_id'])]['owner_team_name']})
    
    #We need to slow this down to not make too many calls in a row but trying to do that less frequently
    if cnt/50 == math.floor(cnt/50):
        
        #We need to slow this down to not make too many calls in a row but trying to do that less frequently
        time.sleep(30)
        
        #let the user know what's going on
        print("Still going ... we are at loop number {} of {}".format(cnt, len(p))) #see how we are doing
        
#convert the list to a dataframe
dfOA = pd.DataFrame.from_records(ls_o)

#Let the user know we got this far
print("Done with Owned Players/ Owner Mapping")

Still going ... we are at loop number 50 of 260
Still going ... we are at loop number 100 of 260
Still going ... we are at loop number 150 of 260
Still going ... we are at loop number 200 of 260
Still going ... we are at loop number 250 of 260
Done with Owned Players/ Owner Mapping


In [18]:
#Union each of the two player dataframes together

#Append the FA pitcher data with the owned pitcher data
#df_pitcher_all = df_fa_p.append(dfOA[dfOA['position_type'] == 'P']).reset_index()  #OLD - trying to get away from .append because it is deprecating
df_pitcher_all = pd.concat([df_fa_p, dfOA[dfOA['position_type'] == 'P']]).reset_index()

#Append the FA batter data with the owned batter data
#df_batter_all = df_fa_b.append(dfOA[dfOA['position_type'] == 'B']).reset_index()    #OLD - trying to get away from .append because it is deprecating
df_batter_all = pd.concat([df_fa_b, dfOA[dfOA['position_type'] == 'B']]).reset_index()

#rename the player_id column to yahoo_id
df_pitcher_all.rename(columns={'player_id': 'yahoo_id'}, inplace=True)
df_batter_all.rename(columns={'player_id': 'yahoo_id'}, inplace=True)

#print a sample to see what it looks like
print(df_batter_all.head())

   index  yahoo_id               name editorial_team_abbr position_type  \
0    218      8634  Giancarlo Stanton                 NYY             B   
1    219      8967   Paul Goldschmidt                 NYY             B   
2    220      8996        Jose Altuve                 HOU             B   
3    221      9096    Travis d'Arnaud                 LAA             B   
4    222      9097       Gary Sánchez                 MIL             B   

       eligible_positions  percent_owned status owner  
0  [LF, RF, OF, Util, IL]             14   IL60    FA  
1          [1B, IF, Util]              5           FA  
2  [2B, IF, LF, OF, Util]             78           FA  
3               [C, Util]              0           FA  
4       [C, 1B, IF, Util]              1           FA  


In [19]:
df_pitcher_all.head()

,index,yahoo_id,name,editorial_team_abbr,position_type,eligible_positions,percent_owned,status,owner
0,0,7590,Justin Verlander,DET,P,[SP],5,,FA
1,1,8193,Max Scherzer,TOR,P,[SP],21,,FA
2,2,8652,Martín Pérez,ATL,P,"[SP, IL]",30,IL15,FA
3,3,9007,Nathan Eovaldi,TEX,P,[SP],70,,FA
4,4,9123,Jameson Taillon,TOR,P,"[SP, IL]",4,IL15,FA


In [20]:
#Step 4 - join the pitcher data together, output the result

#use this function to pull the ID's of pitchers we want to grab data for
df_pitcher_all_f = YahooPlayerMap(df_pitcher_all,yr)

#join the two data sets on the fangraphs ID
df_pitcher_all_f = df_pitcher_all_f.merge(df_pitcher_stat_final, how='left',left_on='IDmlb', right_on='player_id')

#Sort by percent owned so we see the more popular players first
df_pitcher_all_f = df_pitcher_all_f.sort_values(by=['owner','percent_owned'], ascending=False)

#Drop this to Excel so it's easier to read
df_pitcher_all_f.to_excel(output_folder + "Pitchers_All.xlsx", freeze_panes=(1, 3), index=False) #Removed timestamp portion: " + "_" + datetime.today().strftime('%Y%m%d') + "

#Let the user know we are done with this step
print("Pitchers done. Victory!", df_pitcher_all.shape)

Gathering player lookup table. This may take a moment.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
Pitchers done. Victory! (346, 9)


In [21]:
#Step 5 - join the batter data together, output the result

#use this function to pull the ID's of pitchers we want to grab data for
df_batter_all_f = YahooPlayerMap(df_batter_all,yr)

#join the two data sets on the fangraphs ID
df_batter_all_f = df_batter_all_f.merge(df_batter_stat_final, how='left',left_on='IDmlb', right_on='player_id')

#Sort by percent owned so we see the more popular players first
df_batter_all_f = df_batter_all_f.sort_values(by=['owner','percent_owned'], ascending=False)

#Drop this to Excel so it's easier to read
df_batter_all_f.to_excel(output_folder + "Batters_All.xlsx", freeze_panes=(1, 3), index=False) #Removed timestamp portion: " + "_" + datetime.today().strftime('%Y%m%d') + "

#Let the user know we are done with this step
print("Batters done. Victory!", df_batter_all_f.shape)

No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar

In [ ]:
pb.playerid_lookup('rodriguez','grayson',fuzzy=True)

,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,rodriguez,grayson,680570,rodrg003,rodrigr01,24492,2023.0,2026.0


: 